In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import OUTPUT_ROOT

print("OUTPUT_ROOT:", OUTPUT_ROOT)

print("\nContents of OUTPUT_ROOT:")
for p in sorted(OUTPUT_ROOT.iterdir()):
    print("-", p.name)

sample_index_path = OUTPUT_ROOT / "sample_index.csv"
print("\nsample_index.csv exists:", sample_index_path.exists())

if sample_index_path.exists():
    sample_index_df = pd.read_csv(sample_index_path)
    print("\nShape:", sample_index_df.shape)
    print("\nColumns:")
    print(sample_index_df.columns.tolist())
    print("\nFirst 5 rows:")
    display(sample_index_df.head())

OUTPUT_ROOT: /Users/koushik/projects/eeg-seizure-detection-brain-signal-analysis/data/processed

Contents of OUTPUT_ROOT:
- channels.json
- config.json
- file_summary.csv
- metadata
- phase2
- sample_index.csv
- skipped_files.csv
- windows

sample_index.csv exists: True

Shape: (555579, 11)

Columns:
['patient_id', 'file_name', 'window_idx', 'start_sec', 'end_sec', 'label', 'x_path', 'y_path', 'sfreq', 'n_channels', 'n_timepoints']

First 5 rows:


,patient_id,file_name,window_idx,start_sec,end_sec,label,x_path,y_path,sfreq,n_channels,n_timepoints
0,chb01,chb01_01.edf,0,0.0,5.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280
1,chb01,chb01_01.edf,1,2.0,7.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280
2,chb01,chb01_01.edf,2,4.0,9.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280
3,chb01,chb01_01.edf,3,6.0,11.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280
4,chb01,chb01_01.edf,4,8.0,13.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280


In [2]:
import numpy as np
from pathlib import Path

sample_index_df = pd.read_csv(OUTPUT_ROOT / "sample_index.csv")

row = sample_index_df.iloc[0]

print("Selected row:")
display(row.to_frame().T)

x_path = PROJECT_ROOT / row["x_path"]
y_path = PROJECT_ROOT / row["y_path"]
window_idx = int(row["window_idx"])

print("Resolved x_path:", x_path)
print("Resolved y_path:", y_path)
print("window_idx:", window_idx)

print("\nFile existence:")
print("x exists:", x_path.exists())
print("y exists:", y_path.exists())

X_file = np.load(x_path)
y_file = np.load(y_path)

print("\nLoaded file shapes:")
print("X_file shape:", X_file.shape)
print("y_file shape:", y_file.shape)

x_sample = X_file[window_idx]
y_sample = y_file[window_idx]

print("\nSingle sample check:")
print("x_sample shape:", x_sample.shape)
print("y_sample:", y_sample)

print("\nMetadata comparison:")
print("label from sample_index:", int(row["label"]))
print("label from y_file:", int(y_sample))
print("n_channels from sample_index:", int(row["n_channels"]))
print("n_timepoints from sample_index:", int(row["n_timepoints"]))

Selected row:


,patient_id,file_name,window_idx,start_sec,end_sec,label,x_path,y_path,sfreq,n_channels,n_timepoints
0,chb01,chb01_01.edf,0,0.0,5.0,0,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,256.0,18,1280


Resolved x_path: /Users/koushik/projects/eeg-seizure-detection-brain-signal-analysis/data/processed/windows/chb01/chb01_01_X.npy
Resolved y_path: /Users/koushik/projects/eeg-seizure-detection-brain-signal-analysis/data/processed/windows/chb01/chb01_01_y.npy
window_idx: 0

File existence:
x exists: True
y exists: True

Loaded file shapes:
X_file shape: (1798, 18, 1280)
y_file shape: (1798,)

Single sample check:
x_sample shape: (18, 1280)
y_sample: 0

Metadata comparison:
label from sample_index: 0
label from y_file: 0
n_channels from sample_index: 18
n_timepoints from sample_index: 1280


In [3]:
assert X_file.ndim == 3, f"Expected X_file to be 3D, got {X_file.ndim}D"
assert y_file.ndim == 1, f"Expected y_file to be 1D, got {y_file.ndim}D"
assert x_sample.shape[0] == int(row["n_channels"]), "Channel count mismatch"
assert x_sample.shape[1] == int(row["n_timepoints"]), "Timepoint count mismatch"
assert int(row["label"]) == int(y_sample), "Label mismatch between sample_index and y_file"

print("Step 2 verification passed.")

Step 2 verification passed.


In [4]:
import numpy as np
import pandas as pd

sample_index_df = pd.read_csv(OUTPUT_ROOT / "sample_index.csv")

rng = np.random.default_rng(42)
random_indices = rng.choice(len(sample_index_df), size=5, replace=False)

print("Random row indices:", random_indices)

for i, idx in enumerate(random_indices, start=1):
    row = sample_index_df.iloc[idx]

    x_path = PROJECT_ROOT / row["x_path"]
    y_path = PROJECT_ROOT / row["y_path"]
    window_idx = int(row["window_idx"])

    X_file = np.load(x_path)
    y_file = np.load(y_path)

    x_sample = X_file[window_idx]
    y_sample = y_file[window_idx]

    print(f"\n--- Check {i} ---")
    print("row idx:", idx)
    print("patient_id:", row["patient_id"])
    print("file_name:", row["file_name"])
    print("window_idx:", window_idx)
    print("x_sample shape:", x_sample.shape)
    print("label in sample_index:", int(row["label"]))
    print("label in y_file:", int(y_sample))

    assert X_file.ndim == 3, f"Expected X_file to be 3D, got {X_file.ndim}D"
    assert y_file.ndim == 1, f"Expected y_file to be 1D, got {y_file.ndim}D"
    assert x_sample.shape[0] == int(row["n_channels"]), "Channel count mismatch"
    assert x_sample.shape[1] == int(row["n_timepoints"]), "Timepoint count mismatch"
    assert int(row["label"]) == int(y_sample), "Label mismatch"

print("\nStep 3 verification passed.")

Random row indices: [429991 243831 363664  49585 240574]

--- Check 1 ---
row idx: 429991
patient_id: chb15
file_name: chb15_06.edf
window_idx: 1431
x_sample shape: (18, 1280)
label in sample_index: 0
label in y_file: 0

--- Check 2 ---
row idx: 243831
patient_id: chb06
file_name: chb06_17.edf
window_idx: 5782
x_sample shape: (18, 1280)
label in sample_index: 0
label in y_file: 0

--- Check 3 ---
row idx: 363664
patient_id: chb10
file_name: chb10_27.edf
window_idx: 2907
x_sample shape: (18, 1280)
label in sample_index: 0
label in y_file: 0

--- Check 4 ---
row idx: 49585
patient_id: chb01
file_name: chb01_31.edf
window_idx: 48
x_sample shape: (18, 1280)
label in sample_index: 0
label in y_file: 0

--- Check 5 ---
row idx: 240574
patient_id: chb06
file_name: chb06_17.edf
window_idx: 2525
x_sample shape: (18, 1280)
label in sample_index: 0
label in y_file: 0

Step 3 verification passed.


In [5]:
import pandas as pd
import numpy as np

sample_index_df = pd.read_csv(OUTPUT_ROOT / "sample_index.csv")

patient_ids = sorted(sample_index_df["patient_id"].unique().tolist())

print("Total unique patients:", len(patient_ids))
print("Patients:")
print(patient_ids)

Total unique patients: 8
Patients:
['chb01', 'chb02', 'chb06', 'chb08', 'chb10', 'chb12', 'chb15', 'chb18']


In [6]:
rng = np.random.default_rng(42)

shuffled_patients = patient_ids.copy()
rng.shuffle(shuffled_patients)

n = len(shuffled_patients)
n_train = int(round(0.70 * n))
n_val = int(round(0.15 * n))

train_patients = sorted(shuffled_patients[:n_train])
val_patients = sorted(shuffled_patients[n_train:n_train + n_val])
test_patients = sorted(shuffled_patients[n_train + n_val:])

print("Train patients:", train_patients)
print("Val patients:", val_patients)
print("Test patients:", test_patients)

print("\nCounts:")
print("Train:", len(train_patients))
print("Val:", len(val_patients))
print("Test:", len(test_patients))
print("Total:", len(train_patients) + len(val_patients) + len(test_patients))

Train patients: ['chb02', 'chb06', 'chb08', 'chb10', 'chb15', 'chb18']
Val patients: ['chb12']
Test patients: ['chb01']

Counts:
Train: 6
Val: 1
Test: 1
Total: 8


In [7]:
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

assert train_set.isdisjoint(val_set), "Overlap between train and val"
assert train_set.isdisjoint(test_set), "Overlap between train and test"
assert val_set.isdisjoint(test_set), "Overlap between val and test"

print("Step 4 verification passed. No patient overlap.")

Step 4 verification passed. No patient overlap.


In [8]:
import pandas as pd

sample_index_df = pd.read_csv(OUTPUT_ROOT / "sample_index.csv")

train_df = sample_index_df[sample_index_df["patient_id"].isin(train_patients)].copy().reset_index(drop=True)
val_df = sample_index_df[sample_index_df["patient_id"].isin(val_patients)].copy().reset_index(drop=True)
test_df = sample_index_df[sample_index_df["patient_id"].isin(test_patients)].copy().reset_index(drop=True)

print("Train df shape:", train_df.shape)
print("Val df shape:", val_df.shape)
print("Test df shape:", test_df.shape)
print("Total rows:", len(train_df) + len(val_df) + len(test_df))
print("Original rows:", len(sample_index_df))

Train df shape: (445471, 11)
Val df shape: (37197, 11)
Test df shape: (72911, 11)
Total rows: 555579
Original rows: 555579


In [9]:
assert len(train_df) + len(val_df) + len(test_df) == len(sample_index_df), "Row count mismatch after split"

assert set(train_df["patient_id"].unique()).issubset(set(train_patients))
assert set(val_df["patient_id"].unique()).issubset(set(val_patients))
assert set(test_df["patient_id"].unique()).issubset(set(test_patients))

print("Step 5 verification passed.")

Step 5 verification passed.


In [10]:
def summarize_split(df, name):
    positives = int(df["label"].sum())
    negatives = len(df) - positives
    print(f"{name}:")
    print("  rows:", len(df))
    print("  positives:", positives)
    print("  negatives:", negatives)
    print("  patients:", sorted(df["patient_id"].unique().tolist()))
    print()

summarize_split(train_df, "Train")
summarize_split(val_df, "Val")
summarize_split(test_df, "Test")

Train:
  rows: 445471
  positives: 2104
  negatives: 443367
  patients: ['chb02', 'chb06', 'chb08', 'chb10', 'chb15', 'chb18']

Val:
  rows: 37197
  positives: 547
  negatives: 36650
  patients: ['chb12']

Test:
  rows: 72911
  positives: 234
  negatives: 72677
  patients: ['chb01']



In [11]:
import numpy as np
import pandas as pd

def load_split_data_grouped(split_df, max_rows=None):
    """
    Efficiently load samples by grouping rows by x_path/y_path,
    so each file is loaded only once.
    """
    if max_rows is not None:
        split_df = split_df.iloc[:max_rows].copy()

    X_list = []
    y_list = []

    grouped = split_df.groupby(["x_path", "y_path"], sort=False)

    for (x_rel, y_rel), group in grouped:
        x_path = PROJECT_ROOT / x_rel
        y_path = PROJECT_ROOT / y_rel

        window_indices = group["window_idx"].to_numpy(dtype=int)

        X_file = np.load(x_path, mmap_mode="r")
        y_file = np.load(y_path, mmap_mode="r")

        X_chunk = np.array(X_file[window_indices], dtype=np.float32)
        y_chunk = np.array(y_file[window_indices], dtype=np.int64)

        X_list.append(X_chunk)
        y_list.append(y_chunk)

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)

    return X, y

In [12]:
X_small, y_small = load_split_data_grouped(train_df, max_rows=1000)

print("X_small shape:", X_small.shape)
print("y_small shape:", y_small.shape)
print("Positive samples:", int(y_small.sum()))
print("Negative samples:", len(y_small) - int(y_small.sum()))

X_small shape: (1000, 18, 1280)
y_small shape: (1000,)
Positive samples: 0
Negative samples: 1000


In [13]:
print(train_df.head(10)[["patient_id", "file_name", "window_idx", "label"]])
print("Positives in first 1000 rows:", int(train_df.iloc[:1000]["label"].sum()))

  patient_id     file_name  window_idx  label
0      chb02  chb02_01.edf           0      0
1      chb02  chb02_01.edf           1      0
2      chb02  chb02_01.edf           2      0
3      chb02  chb02_01.edf           3      0
4      chb02  chb02_01.edf           4      0
5      chb02  chb02_01.edf           5      0
6      chb02  chb02_01.edf           6      0
7      chb02  chb02_01.edf           7      0
8      chb02  chb02_01.edf           8      0
9      chb02  chb02_01.edf           9      0
Positives in first 1000 rows: 0


In [14]:
train_df_random = train_df.sample(n=1000, random_state=42).reset_index(drop=True)

X_small, y_small = load_split_data_grouped(train_df_random)

print("X_small shape:", X_small.shape)
print("y_small shape:", y_small.shape)
print("Positive samples:", int(y_small.sum()))
print("Negative samples:", len(y_small) - int(y_small.sum()))

X_small shape: (1000, 18, 1280)
y_small shape: (1000,)
Positive samples: 5
Negative samples: 995


In [15]:
train_pos = train_df[train_df["label"] == 1].sample(n=100, random_state=42)
train_neg = train_df[train_df["label"] == 0].sample(n=900, random_state=42)

train_df_mini = pd.concat([train_pos, train_neg], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

X_small, y_small = load_split_data_grouped(train_df_mini)

print("X_small shape:", X_small.shape)
print("y_small shape:", y_small.shape)
print("Positive samples:", int(y_small.sum()))
print("Negative samples:", len(y_small) - int(y_small.sum()))

X_small shape: (1000, 18, 1280)
y_small shape: (1000,)
Positive samples: 100
Negative samples: 900


In [16]:
def build_file_level_summary(split_df, split_name):
    summary = (
        split_df.groupby(
            ["patient_id", "file_name", "x_path", "y_path"],
            as_index=False
        )
        .agg(
            n_windows=("window_idx", "count"),
            n_positive=("label", "sum"),
            min_window_idx=("window_idx", "min"),
            max_window_idx=("window_idx", "max"),
        )
    )

    summary["n_negative"] = summary["n_windows"] - summary["n_positive"]
    summary["split"] = split_name

    return summary


train_file_summary = build_file_level_summary(train_df, "train")
val_file_summary = build_file_level_summary(val_df, "val")
test_file_summary = build_file_level_summary(test_df, "test")

print("Train file summary shape:", train_file_summary.shape)
print("Val file summary shape:", val_file_summary.shape)
print("Test file summary shape:", test_file_summary.shape)

Train file summary shape: (175, 10)
Val file summary shape: (21, 10)
Test file summary shape: (42, 10)


In [17]:
display(train_file_summary.head())
display(val_file_summary.head())
display(test_file_summary.head())


,patient_id,file_name,x_path,y_path,n_windows,n_positive,min_window_idx,max_window_idx,n_negative,split
0,chb02,chb02_01.edf,data/processed/windows/chb02/chb02_01_X.npy,data/processed/windows/chb02/chb02_01_y.npy,1798,0,0,1797,1798,train
1,chb02,chb02_02.edf,data/processed/windows/chb02/chb02_02_X.npy,data/processed/windows/chb02/chb02_02_y.npy,1798,0,0,1797,1798,train
2,chb02,chb02_03.edf,data/processed/windows/chb02/chb02_03_X.npy,data/processed/windows/chb02/chb02_03_y.npy,1798,0,0,1797,1798,train
3,chb02,chb02_04.edf,data/processed/windows/chb02/chb02_04_X.npy,data/processed/windows/chb02/chb02_04_y.npy,1798,0,0,1797,1798,train
4,chb02,chb02_05.edf,data/processed/windows/chb02/chb02_05_X.npy,data/processed/windows/chb02/chb02_05_y.npy,1798,0,0,1797,1798,train


,patient_id,file_name,x_path,y_path,n_windows,n_positive,min_window_idx,max_window_idx,n_negative,split
0,chb12,chb12_06.edf,data/processed/windows/chb12/chb12_06_X.npy,data/processed/windows/chb12/chb12_06_y.npy,1801,50,0,1800,1751,val
1,chb12,chb12_08.edf,data/processed/windows/chb12/chb12_08_X.npy,data/processed/windows/chb12/chb12_08_y.npy,1798,49,0,1797,1749,val
2,chb12,chb12_09.edf,data/processed/windows/chb12/chb12_09_X.npy,data/processed/windows/chb12/chb12_09_y.npy,1804,36,0,1803,1768,val
3,chb12,chb12_10.edf,data/processed/windows/chb12/chb12_10_X.npy,data/processed/windows/chb12/chb12_10_y.npy,1804,42,0,1803,1762,val
4,chb12,chb12_11.edf,data/processed/windows/chb12/chb12_11_X.npy,data/processed/windows/chb12/chb12_11_y.npy,1214,20,0,1213,1194,val


,patient_id,file_name,x_path,y_path,n_windows,n_positive,min_window_idx,max_window_idx,n_negative,split
0,chb01,chb01_01.edf,data/processed/windows/chb01/chb01_01_X.npy,data/processed/windows/chb01/chb01_01_y.npy,1798,0,0,1797,1798,test
1,chb01,chb01_02.edf,data/processed/windows/chb01/chb01_02_X.npy,data/processed/windows/chb01/chb01_02_y.npy,1798,0,0,1797,1798,test
2,chb01,chb01_03.edf,data/processed/windows/chb01/chb01_03_X.npy,data/processed/windows/chb01/chb01_03_y.npy,1798,22,0,1797,1776,test
3,chb01,chb01_04.edf,data/processed/windows/chb01/chb01_04_X.npy,data/processed/windows/chb01/chb01_04_y.npy,1798,15,0,1797,1783,test
4,chb01,chb01_05.edf,data/processed/windows/chb01/chb01_05_X.npy,data/processed/windows/chb01/chb01_05_y.npy,1798,0,0,1797,1798,test


In [18]:
def summarize_file_summary(df, name):
    print(f"{name}:")
    print("  files:", len(df))
    print("  total windows:", int(df["n_windows"].sum()))
    print("  total positives:", int(df["n_positive"].sum()))
    print("  total negatives:", int(df["n_negative"].sum()))
    print()

summarize_file_summary(train_file_summary, "Train")
summarize_file_summary(val_file_summary, "Val")
summarize_file_summary(test_file_summary, "Test")


Train:
  files: 175
  total windows: 445471
  total positives: 2104
  total negatives: 443367

Val:
  files: 21
  total windows: 37197
  total positives: 547
  total negatives: 36650

Test:
  files: 42
  total windows: 72911
  total positives: 234
  total negatives: 72677



In [19]:
assert int(train_file_summary["n_windows"].sum()) == len(train_df)
assert int(val_file_summary["n_windows"].sum()) == len(val_df)
assert int(test_file_summary["n_windows"].sum()) == len(test_df)

assert int(train_file_summary["n_positive"].sum()) == int(train_df["label"].sum())
assert int(val_file_summary["n_positive"].sum()) == int(val_df["label"].sum())
assert int(test_file_summary["n_positive"].sum()) == int(test_df["label"].sum())

print("Step 7 verification passed.")


Step 7 verification passed.


In [ ]:
def compute_train_channel_stats_incremental(file_summary_df, project_root):
    channel_sum = None
    channel_sumsq = None
    total_count = 0

    for _, row in file_summary_df.iterrows():
        x_path = project_root / row["x_path"]

        X_file = np.load(x_path, mmap_mode="r")  # (N, C, T)

        # Cast ONCE safely
        X = X_file.astype(np.float64)

        file_sum = X.sum(axis=(0, 2))                 # (C,)
        file_sumsq = (X ** 2).sum(axis=(0, 2))        # (C,)
        file_count = X.shape[0] * X.shape[2]

        if channel_sum is None:
            channel_sum = file_sum
            channel_sumsq = file_sumsq
        else:
            channel_sum += file_sum
            channel_sumsq += file_sumsq

        total_count += file_count

    mean = channel_sum / total_count
    var = (channel_sumsq / total_count) - (mean ** 2)
    var = np.maximum(var, 1e-12)
    std = np.sqrt(var)

    return mean.astype(np.float32), std.astype(np.float32), total_count

In [23]:
train_mean, train_std, train_count = compute_train_channel_stats_incremental(
    train_file_summary,
    PROJECT_ROOT
)

print("train_mean shape:", train_mean.shape)
print("train_std shape:", train_std.shape)

print("\nFirst few channel means:")
print(np.round(train_mean[:5], 4))

print("\nFirst few channel stds:")
print(np.round(train_std[:5], 4))

train_mean shape: (18,)
train_std shape: (18,)

First few channel means:
[ 0. -0. -0. -0.  0.]

First few channel stds:
[0.0001 0.0001 0.0001 0.0001 0.0001]


In [24]:
print("Min std:", train_std.min())
print("Max std:", train_std.max())

Min std: 5.663766e-05
Max std: 8.120225e-05


In [25]:
print("Std in volts:", train_std[:5])
print("Std in microvolts:", train_std[:5] * 1e6)
print("Min std in microvolts:", train_std.min() * 1e6)
print("Max std in microvolts:", train_std.max() * 1e6)

Std in volts: [6.1066377e-05 6.6567205e-05 5.6637658e-05 6.2285864e-05 8.1202248e-05]
Std in microvolts: [61.066376 66.56721  56.637657 62.285866 81.20225 ]
Min std in microvolts: 56.637657
Max std in microvolts: 81.20225


In [26]:
train_pos = train_df[train_df["label"] == 1].sample(n=100, random_state=42)
train_neg = train_df[train_df["label"] == 0].sample(n=900, random_state=42)

train_df_mini = (
    pd.concat([train_pos, train_neg], axis=0)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [27]:
X_small, y_small = load_split_data_grouped(train_df_mini)

print("Before normalization:")
print("Shape:", X_small.shape)
print("Mean:", X_small.mean())
print("Std:", X_small.std())

Before normalization:
Shape: (1000, 18, 1280)
Mean: 4.9622823e-08
Std: 7.105208e-05


In [28]:
def apply_channelwise_zscore(X, mean, std):
    return ((X - mean[None, :, None]) / std[None, :, None]).astype(np.float32)

X_small_norm = apply_channelwise_zscore(X_small, train_mean, train_std)

In [29]:
print("\nAfter normalization:")
print("Mean:", X_small_norm.mean())
print("Std:", X_small_norm.std())


After normalization:
Mean: 0.0007550448
Std: 1.0714302


In [30]:
channel_means = X_small_norm.mean(axis=(0, 2))
channel_stds = X_small_norm.std(axis=(0, 2))

print("\nPer-channel means (first 5):")
print(np.round(channel_means[:5], 3))

print("\nPer-channel stds (first 5):")
print(np.round(channel_stds[:5], 3))


Per-channel means (first 5):
[ 0.002  0.001 -0.     0.     0.001]

Per-channel stds (first 5):
[1.084 1.07  1.1   1.074 1.049]


In [31]:
assert X_small_norm.shape == X_small.shape
assert np.all(np.isfinite(X_small_norm)), "Non-finite values after normalization"

print("Step 9 verification passed.")

Step 9 verification passed.


In [32]:
from pathlib import Path

phase2_dir = OUTPUT_ROOT / "phase2"
phase2_dir.mkdir(exist_ok=True)

def save_split_incremental(split_df, split_name, mean, std):
    split_dir = phase2_dir / split_name
    split_dir.mkdir(exist_ok=True)

    grouped = split_df.groupby(["x_path", "y_path"], sort=False)

    for (x_rel, y_rel), group in grouped:
        x_path = PROJECT_ROOT / x_rel
        y_path = PROJECT_ROOT / y_rel

        X_file = np.load(x_path, mmap_mode="r")
        y_file = np.load(y_path, mmap_mode="r")

        idxs = group["window_idx"].to_numpy(dtype=int)

        X_chunk = np.array(X_file[idxs], dtype=np.float32)
        y_chunk = np.array(y_file[idxs], dtype=np.int64)

        # normalize
        X_chunk = (X_chunk - mean[None, :, None]) / std[None, :, None]

        # save
        base_name = Path(x_rel).stem.replace("_X", "")
        out_x = split_dir / f"{base_name}_X.npy"
        out_y = split_dir / f"{base_name}_y.npy"

        np.save(out_x, X_chunk)
        np.save(out_y, y_chunk)

        print(f"{split_name} saved:", base_name)

In [33]:
save_split_incremental(train_df, "train", train_mean, train_std)

train saved: chb02_01
train saved: chb02_02
train saved: chb02_03
train saved: chb02_04
train saved: chb02_05
train saved: chb02_06
train saved: chb02_07
train saved: chb02_08
train saved: chb02_09
train saved: chb02_10
train saved: chb02_11
train saved: chb02_12
train saved: chb02_13
train saved: chb02_14
train saved: chb02_15
train saved: chb02_16
train saved: chb02_16+
train saved: chb02_17
train saved: chb02_18
train saved: chb02_19
train saved: chb02_20
train saved: chb02_21
train saved: chb02_22
train saved: chb02_23
train saved: chb02_24
train saved: chb02_25
train saved: chb02_26
train saved: chb02_27
train saved: chb02_28
train saved: chb02_29
train saved: chb02_30
train saved: chb02_31
train saved: chb02_32
train saved: chb02_33
train saved: chb02_34
train saved: chb02_35
train saved: chb06_01
train saved: chb06_02
train saved: chb06_03
train saved: chb06_04
train saved: chb06_05
train saved: chb06_06
train saved: chb06_07
train saved: chb06_08
train saved: chb06_09
train sav

In [34]:
save_split_incremental(val_df, "val", train_mean, train_std)

val saved: chb12_06
val saved: chb12_08
val saved: chb12_09
val saved: chb12_10
val saved: chb12_11
val saved: chb12_19
val saved: chb12_20
val saved: chb12_21
val saved: chb12_23
val saved: chb12_24
val saved: chb12_32
val saved: chb12_33
val saved: chb12_34
val saved: chb12_35
val saved: chb12_36
val saved: chb12_37
val saved: chb12_38
val saved: chb12_39
val saved: chb12_40
val saved: chb12_41
val saved: chb12_42


In [35]:
save_split_incremental(test_df, "test", train_mean, train_std)

test saved: chb01_01
test saved: chb01_02
test saved: chb01_03
test saved: chb01_04
test saved: chb01_05
test saved: chb01_06
test saved: chb01_07
test saved: chb01_08
test saved: chb01_09
test saved: chb01_10
test saved: chb01_11
test saved: chb01_12
test saved: chb01_13
test saved: chb01_14
test saved: chb01_15
test saved: chb01_16
test saved: chb01_17
test saved: chb01_18
test saved: chb01_19
test saved: chb01_20
test saved: chb01_21
test saved: chb01_22
test saved: chb01_23
test saved: chb01_24
test saved: chb01_25
test saved: chb01_26
test saved: chb01_27
test saved: chb01_29
test saved: chb01_30
test saved: chb01_31
test saved: chb01_32
test saved: chb01_33
test saved: chb01_34
test saved: chb01_36
test saved: chb01_37
test saved: chb01_38
test saved: chb01_39
test saved: chb01_40
test saved: chb01_41
test saved: chb01_42
test saved: chb01_43
test saved: chb01_46


In [36]:
import os

for split in ["train", "val", "test"]:
    split_dir = phase2_dir / split
    files = list(split_dir.glob("*_X.npy"))

    print(f"{split}:")
    print("  files:", len(files))

train:
  files: 175
val:
  files: 21
test:
  files: 42


In [37]:
import json
import numpy as np

# save split patient lists
split_info = {
    "train_patients": train_patients,
    "val_patients": val_patients,
    "test_patients": test_patients,
}

with open(phase2_dir / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

# save normalization stats
np.save(phase2_dir / "train_channel_mean.npy", train_mean)
np.save(phase2_dir / "train_channel_std.npy", train_std)

# save row-level metadata per split
train_df.to_csv(phase2_dir / "train_sample_index.csv", index=False)
val_df.to_csv(phase2_dir / "val_sample_index.csv", index=False)
test_df.to_csv(phase2_dir / "test_sample_index.csv", index=False)

print("Saved Phase 2 metadata and normalization stats.")

Saved Phase 2 metadata and normalization stats.
